# Runtime foundations and experimental shared memory

Install `pip install -e '.[ar,notebooks]'`. These CPU cells use the public library only.

Runtime foundations are the Session's typed state, supported writes, recipient-bound commit, rollback, and durable custody. `Program` binds already measured Acts to an exact parent/context; `DependencyGraph` tracks declared dirty dependencies. There is no production permission service or full private runtime in this release.

The memory implementation shares one actual causal KV ancestor across branches, with private appended tails and a global-softmax segmented consumer. It does not implement CUDA virtual-address aliasing, eviction/page faults, dense unlimited context, or independently compiled page composition.

In [ ]:
import json
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/saturn_pub").is_dir())
OUTPUT = ROOT / "outputs/notebooks"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Checkout:", ROOT.name)

In [ ]:
from saturn_pub.adapters.qwen import QwenAdapter
from saturn_pub.experimental.memory import SharedQwenMemory

adapter = QwenAdapter.tiny(seed=7)
prefix = [5, 7, 11]
memory = SharedQwenMemory(adapter, prefix, capacity=8)
left, right = memory.root.fork(), memory.root.fork()
assert left.ancestor_pointers == right.ancestor_pointers
print("One physically shared ancestor:", left.ancestor_pointers == right.ancestor_pointers)

## Consume branch-local history and compare with native execution

Different suffix tokens are supplied here. The following token in each branch is generated from that branch's own predicted logits. Sharing storage does not reduce attention-row work.

In [ ]:
errors = []
for cache, token in ((left, 13), (right, 17)):
    logits = memory.step(cache, token)
    reference = adapter.native_logits(prefix + [token])
    error = float((logits - reference).abs().max())
    errors.append(error)
    predicted = int(logits.argmax())
    memory.step(cache, predicted)
    print(
        {
            "supplied_token": token,
            "native_max_logit_error": error,
            "next_model_predicted_token": predicted,
        }
    )
print(json.dumps(memory.accounting([left, right]), indent=2))
memory.root.verify()

## Branch ownership and immutable ancestry

Forking a live branch keeps the ancestor shared and copies its committed private tail. Appending to the child must not advance the parent.

In [ ]:
child = left.fork()
before = left.get_seq_length()
memory.step(child, 19)
assert left.get_seq_length() == before
assert child.get_seq_length() == before + 1
assert child.ancestor_pointers == left.ancestor_pointers
memory.root.verify()
print("Parent length:", before, "Child length:", child.get_seq_length())

## Phase transport is a separate numerical reference

`PhaseLaw` demonstrates exact modular integer phase composition. It is not connected to the shared Qwen cache above; that cache retains original post-RoPE positions. A relocated phase reference does not establish model behavior or causal composition of independently compiled histories.

In [ ]:
import numpy as np

from saturn_pub.experimental.phase import PhaseLaw

law = PhaseLaw.from_inv_freq([1.0, 0.01])
for _ in range(64):
    law.shift(512)
fresh = PhaseLaw(law.frequencies, origin=64 * 512)
exact = np.array_equal(law.theta([0, 1, 2]), fresh.theta([0, 1, 2]))
print("Exact integer phase composition:", exact)
assert exact

See `docs/runtime-memory-scope.md` for the precise included/omitted map. These are small mechanics experiments, not a serving acceleration or long-context capability claim.